***Step 1: Tạo SparkSession và load các dataset cần thiết***

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IntrusionDetection") \
    .master("local[2]") \
    .getOrCreate()

spark

#Tập training
df_training = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/Training and Testing Sets/UNSW_NB15_training-set.csv",
    header=True, 
    inferSchema=True)
df_training.count()

#

175341

**-Tập ground truth**: Chứa các thông tin về các cuộc tấn công

In [3]:
df_gt = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/NUSW-NB15_GT.csv",
    header=True,
    inferSchema=True)
df_gt.show(5)
df_gt.columns

+----------+----------+---------------+-------------------+--------+------------+-----------+--------------+----------------+--------------------+--------------------+---+
|Start time| Last time|Attack category| Attack subcategory|Protocol|   Source IP|Source Port|Destination IP|Destination Port|         Attack Name|    Attack Reference|  .|
+----------+----------+---------------+-------------------+--------+------------+-----------+--------------+----------------+--------------------+--------------------+---+
|1421927414|1421927416| Reconnaissance|               HTTP|     tcp|175.45.176.0|      13284|149.171.126.16|              80|Domino Web Server...|                   -|  .|
|1421927415|1421927415|       Exploits|   Unix 'r' Service|     udp|175.45.176.3|      21223|149.171.126.18|           32780|Solaris rwalld Fo...|CVE 2002-0573 (ht...|  .|
|1421927416|1421927416|       Exploits|            Browser|     tcp|175.45.176.2|      23357|149.171.126.16|              80|Windows Metafil

['Start time',
 'Last time',
 'Attack category',
 'Attack subcategory',
 'Protocol',
 'Source IP',
 'Source Port',
 'Destination IP',
 'Destination Port',
 'Attack Name',
 'Attack Reference',
 '.']

**-Đọc các features của datasets:** Bao gồm 49 features khác nhau cho các lượt truy cập mạng

In [4]:
import pandas as pd
features = pd.read_csv("/home/jovyan/intrusion_data/raw/CSV Files/NUSW-NB15_features.csv", encoding="cp1252")
col_names = list(features["Name"])
print(col_names)
print(len(col_names))

['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']
49


**-Đọc các bản ghi gốc và kiểm tra các cột:**

In [5]:
raw_df = spark.read.csv(
    "/home/jovyan/intrusion_data/raw/CSV Files/",
    header=False,
    inferSchema=True
).toDF(*col_names)

In [6]:
print(f"Rows: {raw_df.count()}")
print(f"Columns: {len(raw_df.columns)}")

Rows: 2730540
Columns: 49


**Kiểm tra list events:**


In [7]:
list_event_df = spark.read.csv("/home/jovyan/intrusion_data/raw/CSV Files/UNSW-NB15_LIST_EVENTS.csv",
    header=True,
    inferSchema=True,
    schema=None)

list_event_df.show()

+----------------+--------------------+----------------+
| Attack category|  Attack subcategory|Number of events|
+----------------+--------------------+----------------+
|          normal|                NULL|         2218761|
|        Fuzzers |                 FTP|             558|
|        Fuzzers |                HTTP|            1497|
|        Fuzzers |                 RIP|            3550|
|        Fuzzers |                 SMB|            5245|
|        Fuzzers |              Syslog|            1851|
|        Fuzzers |                PPTP|            1583|
|         Fuzzers|                 FTP|             248|
|         Fuzzers|              DCERPC|             164|
|         Fuzzers|                OSPF|             993|
|        Fuzzers |                TFTP|             193|
|        Fuzzers |             DCERPC |             455|
|        Fuzzers |                OSPF|            1746|
|        Fuzzers |                 BGP|            6163|
| Reconnaissance |             

**Step 2: Load các công cụ cần thiết để khám phá dữ liệu:**

In [8]:
from pyspark.sql.functions import from_unixtime, col,to_timestamp, isnan, when, count, window, trim, regexp_replace

In [9]:

raw_df = raw_df.withColumn(
    "timestamp", 
    from_unixtime(col("Stime"))
)


raw_df = raw_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)
raw_df.filter(col("attack_cat").isNotNull()).count()


321283

In [10]:

numeric_cols = ["sbytes", "dbytes", "dur", "sttl", "dttl", 
                "sloss", "dloss", "sport", "dsport"]

for c in numeric_cols:
    raw_df = raw_df.withColumn(c, col(c).cast("double"))

In [11]:
raw_df.printSchema()

root
 |-- srcip: string (nullable = true)
 |-- sport: double (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: double (nullable = true)
 |-- proto: string (nullable = true)
 |-- state: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: double (nullable = true)
 |-- dbytes: double (nullable = true)
 |-- sttl: double (nullable = true)
 |-- dttl: double (nullable = true)
 |-- sloss: double (nullable = true)
 |-- dloss: double (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: double (nullable = true)
 |-- Dload: double (nullable = true)
 |-- Spkts: integer (nullable = true)
 |-- Dpkts: integer (nullable = true)
 |-- swin: integer (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- stcpb: long (nullable = true)
 |-- dtcpb: long (nullable = true)
 |-- smeansz: integer (nullable = true)
 |-- dmeansz: integer (nullable = true)
 |-- trans_depth: integer (nullable = true)
 |-- res_bdy_len: integer (nullable = true)
 |-- Sjit: dou

In [12]:
raw_df = raw_df.withColumn("attack_cat", trim(col("attack_cat")))


In [13]:
raw_df.groupBy("attack_cat").count().orderBy("count", ascending=False).show()

+--------------+-------+
|    attack_cat|  count|
+--------------+-------+
|          NULL|2409257|
|       Generic| 215481|
|      Exploits|  44525|
|       Fuzzers|  24246|
|           DoS|  16353|
|Reconnaissance|  13987|
|      Analysis|   2677|
|      Backdoor|   1795|
|     Shellcode|   1511|
|     Backdoors|    534|
|         Worms|    174|
+--------------+-------+



In [14]:

raw_df = raw_df.withColumn(
    "attack_cat",
    regexp_replace(col("attack_cat"), "Backdoors", "Backdoor")
)

raw_df.groupBy("attack_cat").count().orderBy("count", ascending=False).show()

+--------------+-------+
|    attack_cat|  count|
+--------------+-------+
|          NULL|2409257|
|       Generic| 215481|
|      Exploits|  44525|
|       Fuzzers|  24246|
|           DoS|  16353|
|Reconnaissance|  13987|
|      Analysis|   2677|
|      Backdoor|   2329|
|     Shellcode|   1511|
|         Worms|    174|
+--------------+-------+



In [15]:
raw_df.select("dur").describe().show()

+-------+-----------------+
|summary|              dur|
+-------+-----------------+
|  count|          2714394|
|   mean|989.1631147190352|
| stddev|6671.036201451125|
|    min|              0.0|
|    max|          65535.0|
+-------+-----------------+



In [16]:
raw_df.filter(col("dur") == 65535.0)\
      .groupBy("attack_cat")\
      .count()\
      .show()

+----------+-----+
|attack_cat|count|
+----------+-----+
|      NULL|   33|
+----------+-----+



In [17]:


raw_df = raw_df.withColumn(
    "bytes_ratio",
    when(col("dbytes") == 0, 0)
    .otherwise(col("sbytes") / col("dbytes"))
)

In [35]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

@udf(DoubleType())
def threat_score(sbytes, avg_sbytes_proto, sload, avg_sload_proto):
    if None in [sbytes, avg_sbytes_proto, sload, avg_sload_proto]:
        return 0.0
    bytes_dev = abs(float(sbytes) - float(avg_sbytes_proto)) / (float(avg_sbytes_proto) + 1)
    load_dev = abs(float(sload) - float(avg_sload_proto)) / (float(avg_sload_proto) + 1)
    return float(bytes_dev * 0.5 + load_dev * 0.5)
     

In [36]:
raw_df = raw_df.withColumn(
    "threat_score",
    threat_score(col("sbytes"), col("avg_sbytes_proto"), col("sload"), col("avg_sload_proto"))
)

In [21]:
from pyspark.sql.functions import udf, avg
raw_df.filter(col("attack_cat") == "DoS")\
      .groupBy("proto")\
      .agg(
          avg("sbytes").alias("avg_sbytes"),
          avg("dur").alias("avg_dur"),
          count("*").alias("count")
      )\
      .orderBy("count", ascending=False)\
      .show(10)

+-----+------------------+--------------------+-----+
|proto|        avg_sbytes|             avg_dur|count|
+-----+------------------+--------------------+-----+
| unas|             200.0|6.920892108272998...| 5246|
|  tcp| 83250.74700239809|   3.143823941846525| 3336|
| ospf| 7868.813559322034|    27.9084453506356|  944|
|  udp| 4315.861480075901|  2.2312447666034148|  527|
| sctp| 24875.74178403756|  3.1564213474178353|  426|
|  any|             200.0|7.825757575757577E-6|  132|
|  gre|174.09345794392524|7.747663551401871E-6|  107|
| rsvp|             200.0|6.840909090909090...|   88|
| ipv6|145.71428571428572|7.571428571428572E-6|   84|
|  sep|154.89156626506025|6.397590361445785E-6|   83|
+-----+------------------+--------------------+-----+
only showing top 10 rows



In [19]:
proto_stats = raw_df.groupBy("proto").agg(
    avg("dur").alias("avg_dur_proto"),
    avg("sbytes").alias("avg_sbytes_proto"),
    avg("dbytes").alias("avg_dbytes_proto"),
    avg("sload").alias("avg_sload_proto"),
    avg("spkts").alias("avg_spkts_proto"),
    avg("dpkts").alias("avg_dpkts_proto")
)

In [26]:
from pyspark.sql.functions import broadcast
raw_df = raw_df.join(broadcast(proto_stats), on='proto', how='left')

In [28]:
state_stat = raw_df.groupBy('state').agg(
    avg('dur').alias("avg_dur_state"),
    avg('sbytes').alias('avg_sbytes_state'),
    avg('dbytes').alias('avg_dbytes_state'),
    avg('sload').alias('avg_sload_state'),
    avg('spkts').alias('avg_spkts_state'),
    avg('dpkts').alias('avg_dpkts_state'),
)
raw_df = raw_df.join(broadcast(state_stat), on='state', how='left')

In [22]:
raw_df.select("attack_cat").distinct().show()

+--------------+
|    attack_cat|
+--------------+
|         Worms|
|     Shellcode|
|       Fuzzers|
|      Analysis|
|           DoS|
|Reconnaissance|
|      Backdoor|
|      Exploits|
|       Generic|
|          NULL|
+--------------+



In [23]:
print(raw_df.filter(col("attack_cat").isNotNull()).count())

321283


In [32]:
raw_df = raw_df.drop('srcip', 'dstip', 'sport', 'dsport', 'stcpb', 'dtcpb', 'Stime', 'Ltime')

In [34]:
print(len(raw_df.columns))
raw_df.printSchema()

56
root
 |-- state: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: double (nullable = true)
 |-- dbytes: double (nullable = true)
 |-- sttl: double (nullable = true)
 |-- dttl: double (nullable = true)
 |-- sloss: double (nullable = true)
 |-- dloss: double (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: double (nullable = true)
 |-- Dload: double (nullable = true)
 |-- Spkts: integer (nullable = true)
 |-- Dpkts: integer (nullable = true)
 |-- swin: integer (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- smeansz: integer (nullable = true)
 |-- dmeansz: integer (nullable = true)
 |-- trans_depth: integer (nullable = true)
 |-- res_bdy_len: integer (nullable = true)
 |-- Sjit: double (nullable = true)
 |-- Djit: double (nullable = true)
 |-- Sintpkt: double (nullable = true)
 |-- Dintpkt: double (nullable = true)
 |-- tcprtt: double (nullable = true)
 |-- synack: double (nullable = true)
 |--

In [25]:

raw_df.groupBy(
    window("timestamp", "5 minutes")
).agg(
    count("*").alias("packet_count")
).orderBy("window").show()

+--------------------+------------+
|              window|packet_count|
+--------------------+------------+
|{2015-01-22 11:45...|           6|
|{2015-01-22 11:50...|        7383|
|{2015-01-22 11:55...|        7424|
|{2015-01-22 12:00...|        8712|
|{2015-01-22 12:05...|        7805|
|{2015-01-22 12:10...|        7575|
|{2015-01-22 12:15...|        6571|
|{2015-01-22 12:20...|        6796|
|{2015-01-22 12:25...|        7506|
|{2015-01-22 12:30...|        7619|
|{2015-01-22 12:35...|        7495|
|{2015-01-22 12:40...|        8597|
|{2015-01-22 12:45...|       10855|
|{2015-01-22 12:50...|        8355|
|{2015-01-22 12:55...|        8403|
|{2015-01-22 13:00...|        8156|
|{2015-01-22 13:05...|        9137|
|{2015-01-22 13:10...|        8699|
|{2015-01-22 13:15...|        8165|
|{2015-01-22 13:20...|        8047|
+--------------------+------------+
only showing top 20 rows



In [37]:
# fix ct_ftp_cmd
raw_df = raw_df.withColumn("ct_ftp_cmd", col("ct_ftp_cmd").cast("integer"))

# drop timestamp (not needed for ML)
raw_df = raw_df.drop("timestamp")

In [38]:
raw_df.count()

2730540

In [39]:
raw_df.printSchema()

root
 |-- state: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: double (nullable = true)
 |-- dbytes: double (nullable = true)
 |-- sttl: double (nullable = true)
 |-- dttl: double (nullable = true)
 |-- sloss: double (nullable = true)
 |-- dloss: double (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: double (nullable = true)
 |-- Dload: double (nullable = true)
 |-- Spkts: integer (nullable = true)
 |-- Dpkts: integer (nullable = true)
 |-- swin: integer (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- smeansz: integer (nullable = true)
 |-- dmeansz: integer (nullable = true)
 |-- trans_depth: integer (nullable = true)
 |-- res_bdy_len: integer (nullable = true)
 |-- Sjit: double (nullable = true)
 |-- Djit: double (nullable = true)
 |-- Sintpkt: double (nullable = true)
 |-- Dintpkt: double (nullable = true)
 |-- tcprtt: double (nullable = true)
 |-- synack: double (nullable = true)
 |-- ac